<a href="https://colab.research.google.com/github/zhesun0304/ECON3916/blob/main/Lab12/Architecting_the_Prediction_Engine_OLS%2C_Hedonic_Pricing%2C_and_RMSE_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Step 1: Environment Initialization and Data Ingestion
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.tools.eval_measures import rmse
import matplotlib.pyplot as plt

# Load the dataset from GitHub raw link
url = "https://raw.githubusercontent.com/zhesun0304/ECON3916/main/data/Zillow_ZHVI_2026_Micro.csv"
df = pd.read_csv(url)

# Quick EDA
print(df.head())
print("\nData shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nSummary statistics:")
print(df.describe())


   Home_Value  Square_Footage  Property_Age  Distance_to_Transit  \
0   329705.74          1941.0           5.5                 6.45   
1   183343.63          1364.3          35.2                 2.15   
2   354551.73          2386.9          52.4                 0.75   
3   325773.17          2192.1          50.2                 5.25   
4   359743.12          3069.8          66.5                12.69   

  School_District_Rating  
0              Excellent  
1                Average  
2                   Good  
3              Excellent  
4              Excellent  

Data shape: (1000, 5)

Columns: ['Home_Value', 'Square_Footage', 'Property_Age', 'Distance_to_Transit', 'School_District_Rating']

Summary statistics:
          Home_Value  Square_Footage  Property_Age  Distance_to_Transit
count    1000.000000     1000.000000     1000.0000          1000.000000
mean   320008.067140     2244.538300       39.7171             7.587540
std     87569.422238      616.243881       23.4256           

In [2]:
# Step 2: Defining the OLS Architecture via Patsy Formula
formula = "Home_Value ~ Square_Footage + Property_Age + Distance_to_Transit"
print("Model Formula:", formula)

Model Formula: Home_Value ~ Square_Footage + Property_Age + Distance_to_Transit


In [3]:
# Step 3: Model Fitting and Diagnostic Extraction
model = smf.ols(formula=formula, data=df)
results = model.fit()
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:             Home_Value   R-squared:                       0.766
Model:                            OLS   Adj. R-squared:                  0.765
Method:                 Least Squares   F-statistic:                     1087.
Date:                Mon, 30 Mar 2026   Prob (F-statistic):          1.73e-313
Time:                        00:46:23   Log-Likelihood:                -12072.
No. Observations:                1000   AIC:                         2.415e+04
Df Residuals:                     996   BIC:                         2.417e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept            8.898e+04   6

In [4]:
# Step 4: Generating Predictions
y_pred = results.predict(df)
print(y_pred.head())

0    312391.414318
1    222863.862825
2    333724.313631
3    307478.039316
4    392700.239719
dtype: float64


In [6]:
# Step 5: Calculating and Interpreting the Root Mean Squared Error
model_rmse = rmse(df["Home_Value"], y_pred)
print(f"\nThe Predictive RMSE is: ${model_rmse:,.2f}")


The Predictive RMSE is: $42,341.60


In [7]:
# Step 6: Residual Forensics Data Preparation

import pandas as pd
import numpy as np

# The statsmodels results object stores the fitted values and residuals directly.
# results.fittedvalues = model-predicted Home_Value for each observation
# results.resid = actual Home_Value - predicted Home_Value for each observation
fitted_vals = results.fittedvalues
residuals = results.resid

# Compute the sample standard deviation of the residuals
resid_sd = residuals.std(ddof=1)

# Flag observations whose residual magnitude exceeds 2 standard deviations
# This is a common visual rule for unusually large prediction errors
outlier_mask = np.abs(residuals) > 2 * resid_sd

# Build a diagnostics DataFrame for Plotly
resid_df = pd.DataFrame({
    "Predicted_Home_Value": fitted_vals,
    "Residual_Error": residuals,
    "Outlier_Flag": np.where(outlier_mask, "Outlier (>2 SD)", "Within ±2 SD")
})

# Optional extra fields for better hover labels
resid_df["Abs_Residual"] = np.abs(resid_df["Residual_Error"])
resid_df["Residual_zscore"] = resid_df["Residual_Error"] / resid_sd

print(resid_df.head())
print(f"Residual standard deviation: {resid_sd:,.2f}")
print(f"Number of flagged outliers: {outlier_mask.sum()}")

   Predicted_Home_Value  Residual_Error  Outlier_Flag  Abs_Residual  \
0         312391.414318    17314.325682  Within ±2 SD  17314.325682   
1         222863.862825   -39520.232825  Within ±2 SD  39520.232825   
2         333724.313631    20827.416369  Within ±2 SD  20827.416369   
3         307478.039316    18295.130684  Within ±2 SD  18295.130684   
4         392700.239719   -32957.119719  Within ±2 SD  32957.119719   

   Residual_zscore  
0         0.408715  
1        -0.932900  
2         0.491644  
3         0.431868  
4        -0.777973  
Residual standard deviation: 42,362.79
Number of flagged outliers: 36
